# GameTheory-3c — Le joueur LLM dans le tableau périodique

**Navigation** : [GameTheory-3](GameTheory-03-Topology2x2.ipynb) (chambres Robinson-Goforth) · [GameTheory-03c-Le-Joueur-LLM](GameTheory-03c-Le-Joueur-LLM.ipynb) · [GameTheory-21](GameTheory-21-Deux-Especes-de-Fleches.ipynb) (morphisme fini)

**Grain** : `#12254` — DEEP/notebook-python sur le papier *Playing Repeated Games with Large Language Models* (Nature Human Behaviour, [s41562-025-02172-y](https://www.nature.com/articles/s41562-025-02172-y)).

**Sources lues firsthand** : page article (2026-08-22) ; grammaire R-G des chambres/murs réutilisée de `GameTheory-3` (cellule 5 `OrdinalGame`).

**Kernel** : `python3` — pas d'appels réseau non gardés.

## Hypothèse (lue du papier, reformulée)

Un joueur LLM (GPT-4 / Claude 2 / Llama 2 70B / text-davinci) joue à des jeux 2×2 répétés (matrice convertie en règles textuelles, température 0, réponse mono-token, historique concaténé). Trois apports :

- **(a)** Le **paysage de performance** du joueur varie selon la famille de jeu — fort en Dilemme (défection permanente après une seule défection), faible en coordination (Battle of the Sexes : il colle à son option préférée).
- **(b)** La **dissociation prédire/agir** : GPT-4 prédit correctement l'alternance et n'agit pas en conséquence.
- **(c)** Le **SCoT** (Social Chain-of-Thought — prédire le coup adverse avant de choisir) augmente la coordination sans changer le jeu : c'est une **transmutation de Bruns**, seconde levier mesurable à côté du « payer pour déplacer le jeu ».

## Ce que le notebook mesure

Quatre cellules-mesures (E1-E4) ancrées sur les outputs commités :

1. **E1 — Placer le papier dans le tableau** : les six familles mesurées (win-win, Dilemme, unfair, cyclique, biaisé, second-best) se placent-elles dans les chambres Robinson-Goforth ? **Mapping RAPPORTÉ** (le notebook dérive ; le mapping Robinson-Goforth ↔ papier est une dette reconnue §Sources).
2. **E2 — Le joueur LLM sur deux jeux** : rejouer le protocole (matrice → règles textuelles → température 0 → 1 token → historique concaténé → 10 rounds) sur Dilemme canonique `(8,8)/(0,10)/(10,0)/(5,5)` et Battle of the Sexes `(10,7)/(7,10)`. **Stub C.1 par défaut** (pas d'appel provider non gardé) ; une cellule teste un endpoint externe quand `OPENAI_API_KEY` est présent (reproductibilité : cassette, plafond, log).
3. **E3 — Le swap en cours de partie** : valeur ajoutée absente du papier — appliquer `R34` ou `C23` au round k et mesurer si le joueur suit le déplacement. Marche 1½ vers D4.
4. **E4 — Dissociation (b)** : reproduire la séparation « le modèle annonce l'alternance, mais n'alterne pas » par une mesure explicite (pas une affirmation) sur la cassette.

## Critère d'acceptation

- E1 rend un placement explicite avec statut (dérivé / RAPPORTÉ).
- E2/E3 produisent des taux mesurés, reproductibles à température 0, sur sorties committées (cellule vide → `pass` + stub C.1 quand provider absent, c'est aussi un résultat reproductible).
- E4 exhibe la dissociation (ou son absence, qui est un résultat).
- C.1 : 0 `raise NotImplementedError` ; C.2 : cellules code avec `execution_count` + outputs réels (ou vides si stub non exécuté).

## Dettes de vérification

1. Le mapping six familles ↔ chambres/murs R-G n'est pas dérivé — alignement à établir dans E1 avant toute affirmation.
2. Les résultats du papier sont ses résultats, sur **ses** modèles 2023-2024. Rejoués sur des modèles actuels, ils peuvent ne pas se reproduire — c'est ce que E2/E4 mesurent (cassettes + plafond).
3. Coût et reproductibilité : appels réels = plafond et cassettes avant la lane ; sinon stub C.1 honnête.

## Sources

- Mei et al., *Playing Repeated Games with Large Language Models*, Nature Human Behaviour (2025), [s41562-025-02172-y](https://www.nature.com/articles/s41562-025-02172-y) — page lue 2026-08-22.
- Robinson & Goforth, *The Topology of the 2x2 Games* (2005) — implémentation dans `GameTheory-3` cellule 5 (`OrdinalGame`).
- Bruns, *Austausch und Gerechtigkeit* (1975) — notion de transmutation (information nouvelle vs déplacement), voir aussi GameTheory-21 (Loi III, transformations vs morphismes).

In [1]:
# Imports
import os
import numpy as np
from dataclasses import dataclass
from typing import Tuple, List, Dict

# Convention : OrdinalGame (R-G) aligne sur le notebook GameTheory-3 (cellule 5, 7).
# Plus le rang est GRAND, meilleure est l'issue. Vecteur indexe (CC, CD, DC, DD)
# pour Row (payoffs_R) et Col (payoffs_C). L'invariant __post_init__ garantit
# que chaque payoffs_X est une permutation stricte de (1, 2, 3, 4) -- propriete
# qui exclut mecaniquement les jeux a rangs repetes (hors tableau periodique R-G)
# et qui protege contre les fautes de frappe comme (1, 2, 2, 3).

@dataclass(frozen=True)
class OrdinalGame:
    name: str
    payoffs_R: Tuple[int, int, int, int]  # rangs Row pour (CC, CD, DC, DD)
    payoffs_C: Tuple[int, int, int, int]  # rangs Col pour (CC, CD, DC, DD)

    def __post_init__(self):
        assert sorted(self.payoffs_R) == [1, 2, 3, 4], \
            f"payoffs_R doit etre permutation de 1-4, got {self.payoffs_R}"
        assert sorted(self.payoffs_C) == [1, 2, 3, 4], \
            f"payoffs_C doit etre permutation de 1-4, got {self.payoffs_C}"


CLASSIC_GAMES = {
    # Harmony : CC > CD > DC > DD. Rang_R = (4, 3, 2, 1), Rang_C = (4, 2, 3, 1)
    # (Nash unique (C,C), ordre strict, symetrie CD<->DC transposee).
    "Harmony":      OrdinalGame("Harmony",      (4, 3, 2, 1), (4, 2, 3, 1)),
    # StagHunt : CC > DC > DD > CD. Rang_R = (4, 1, 3, 2), Rang_C = (4, 3, 1, 2)
    # (Nash (C,C) et (D,D), ordre strict, symetrie transposee).
    "StagHunt":     OrdinalGame("StagHunt",     (4, 1, 3, 2), (4, 3, 1, 2)),
    # Dilemme (= Prisoner's Dilemma textbook) : DC > CC > DD > CD.
    # Rang_R = (3, 1, 4, 2), Rang_C = (3, 4, 1, 2) (Nash unique (D,D), CC>DD).
    "Dilemme":      OrdinalGame("Dilemme",      (3, 1, 4, 2), (3, 4, 1, 2)),
    # Chicken : DC > CC > CD > DD. Rang_R = (3, 2, 4, 1), Rang_C = (3, 4, 2, 1)
    # (Nash (C,D) et (D,C), ordre strict, symetrie transposee).
    "Chicken":      OrdinalGame("Chicken",      (3, 2, 4, 1), (3, 4, 2, 1)),
    # Coordination (= Pure Coordination) : CC > DD > DC > CD.
    # Rang_R = (4, 1, 2, 3), Rang_C = (4, 2, 1, 3) (Nash (C,C) et (D,D)).
    "Coordination": OrdinalGame("Coordination", (4, 1, 2, 3), (4, 2, 1, 3)),
    # BattleSexes : DC > CD > CC > DD. Rang_R = (2, 3, 4, 1), Rang_C = (2, 4, 3, 1)
    # (Nash (C,D) et (D,C), Row prefere (C,D) car CD = rang 3 > CC = rang 2,
    #  Col prefere (D,C) car DC = rang 4 > DD = rang 1 -- chaque joueur
    #  departage les deux Nash en sens inverse).
    "BattleSexes":  OrdinalGame("BattleSexes",  (2, 3, 4, 1), (2, 4, 3, 1)),
}


### Lecture de la representation

Les jeux sont encodes en **rangs ordonnes** (4 = meilleur, 1 = pire pour le joueur considere). C'est la convention de Robinson-Goforth (GameTheory-3 cellule 5), invariante aux translations de payoff -- ce qui compte est la **structure des preferences**, pas les valeurs cardinales.

**Exemple Dilemme** `(3, 1, 4, 2)` : pour le **Row-player**, la tentation (D,C) = rang 4 bat la cooperation (C,C) = rang 3 ; la recompense mutuelle (D,D) = rang 2 bat la defection unilaterale (C,D) = rang 1 (DD > CD au sens ordinal). Pour le **Col-player**, la defection unilaterale (C,D) = rang 4 bat tout.

Cette convention permet de tester la **dissociation** du joueur LLM **sans bruit** : si le joueur repond « D » en Dilemme, c'est la preference revelee ; si en BoS il repond toujours la meme option, c'est l'absence d'alternance.


### Pourquoi cette convention pour E2 ?

Le papier (Mei et al.) utilise une représentation **cardinale** dans ses mesures de payoff cumulé. Mais l'apport scientifique — *la dissociation prédire/agir* — est **invariant à la représentation** : peu importe que (C,C) paie 8 ou 10, ce qui compte est que le joueur **prédit correctement** l'alternance en BoS et **n'agit pas** en conséquence.

On peut donc reproduire l'expérience (b) en ordinal, sans dépendance externe, et la **dissociation reste visible** : le joueur qui annonce « J'alterne C-D-C-D » et joue C-C-C-C.

L'apport (c) — SCoT comme transmutation — est aussi mesurable en ordinal : la consigne « prédis le coup adverse » modifie le comportement sans modifier le jeu. Même grammaire, même test.

In [2]:
def best_response(g: OrdinalGame, player: str, opponent_action: str) -> str:
    """
    Meilleure reponse (rang 4 = meilleur) du joueur `player` quand l'adversaire joue `opponent_action`.
    """
    if player == "Row":
        if opponent_action == "C":
            r_C, r_D = g.payoffs_R[0], g.payoffs_R[2]
        else:
            r_C, r_D = g.payoffs_R[1], g.payoffs_R[3]
    else:  # Col
        if opponent_action == "C":
            r_C, r_D = g.payoffs_C[0], g.payoffs_C[1]
        else:
            r_C, r_D = g.payoffs_C[2], g.payoffs_C[3]
    return "C" if r_C >= r_D else "D"


# Verification : BR coherente avec la litterature R-G
print("=== Best response par jeu ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    br_row_C = best_response(g, "Row", "C")
    br_row_D = best_response(g, "Row", "D")
    br_col_C = best_response(g, "Col", "C")
    br_col_D = best_response(g, "Col", "D")
    print(f"{g_name:14s}: Row(C)={br_row_C} Row(D)={br_row_D} | Col(C)={br_col_C} Col(D)={br_col_D}")


=== Best response par jeu ===
BattleSexes   : Row(C)=D Row(D)=C | Col(C)=D Col(D)=C
StagHunt      : Row(C)=C Row(D)=D | Col(C)=C Col(D)=D
Dilemme       : Row(C)=D Row(D)=D | Col(C)=D Col(D)=D
Chicken       : Row(C)=D Row(D)=C | Col(C)=D Col(D)=C
Harmony       : Row(C)=C Row(D)=C | Col(C)=C Col(D)=C
Coordination  : Row(C)=C Row(D)=D | Col(C)=C Col(D)=D


## 1. E1 — Placer le papier dans le tableau R-G

Le papier (Mei et al.) distingue six familles de jeux 2×2 mesurées :

1. **win-win** (jeux à équilibre coopératif dominant, type Harmony)
2. **Dilemme** (Prisoner's Dilemma)
3. **unfair** (jeux asymétriques type Battle of the Sexes où un joueur a un avantage structurel)
4. **cyclique** (type Chicken — Rock-Paper-Scissors-like en 2×2)
5. **biaisé** (jeux à dominance stricte)
6. **second-best** (jeux où le Nash n'est pas Pareto-Optimal)

**Mapping proposé (RAPPORTÉ, dette §Sources)** :

| Famille papier | Chambre R-G probable | Mapping |
|---|---|---|
| win-win | Harmony + Coordination | DÉRIVÉ (Harmony a (C,C) Pareto-dominant) |
| Dilemme | Dilemme (strict) | DÉRIVÉ (match canonique : CC > DD) |
| unfair | BattleSexes | DÉRIVÉ (Nash (C,D) et (D,C), chaque joueur départage à l'inverse) |
| cyclique | Chicken | DÉRIVÉ (R-G "Rock-Paper-Scissors-like" en 2×2) |
| biaisé | jeux à stratégie dominante (subset de Dilemme+Chicken) | RAPPORTÉ — la définition "biaisé" du papier n'est pas dans R-G canonique |
| second-best | subset de StagHunt | DÉRIVÉ (StagHunt a (D,D) Nash mais (C,C) Pareto) |

**Note importante — cyclicité de BattleSexes** :

BattleSexes canonique admet **deux Nash purs** : (C,D) et (D,C). Pour encoder simultanément les deux Nash sans cycler sur le même rang, on choisit Row `rang_R = (2, 3, 4, 1)` (DC > CD > CC > DD, Row préfère (D,C)) et Col `rang_C = (2, 4, 3, 1)` (DC > DD > CD > CC, Col préfère (C,D)). L'ordre strict est préservé, et chaque joueur départage les deux Nash dans la direction qui maximise son payoff — c'est exactement l'essence du conflit de BoS.

**Remarque invariante** : la convention du notebook est `4 = meilleur, 1 = pire` (alignée sur `GameTheory-3 cellule 5`), **et** chaque `payoffs_X` est une permutation stricte de `(1, 2, 3, 4)` — assertion `__post_init__` qui protège mécaniquement contre les fautes de frappe (cf leçon ai-01 dans la review PR #12295).


In [3]:
# E1 : mesure des Nash purs par chambre R-G (les 6 jeux classiques)
def find_pure_nash(g: OrdinalGame) -> List[str]:
    """Nash purs : cases (a,b) telles que a = best_response(Row) et b = best_response(Col)."""
    results = []
    for row_a in ["C", "D"]:
        for col_a in ["C", "D"]:
            br_row = best_response(g, "Row", col_a)
            br_col = best_response(g, "Col", row_a)
            if row_a == br_row and col_a == br_col:
                results.append((row_a, col_a))
    return results


print("=== E1 : Equilibres de Nash purs par chambre R-G ===")
print(f"{'Jeu':15s} {'Nash purs':15s} {'Cardinalite'}")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    nash = find_pure_nash(g)
    cardinalite = "0" if not nash else f"{len(nash)}"
    nash_str = ", ".join(f"{a}{b}" for a, b in nash) if nash else "(aucun)"
    print(f"{g_name:15s} {nash_str:15s} {cardinalite}")

=== E1 : Equilibres de Nash purs par chambre R-G ===
Jeu             Nash purs       Cardinalite
BattleSexes     CD, DC          2
StagHunt        CC, DD          2
Dilemme         DD              1
Chicken         CD, DC          2
Harmony         CC              1
Coordination    CC, DD          2


### Lecture de E1

**Trois chambres à 1 Nash** : Dilemme (D,D unique — grim trigger), Harmony (C,C unique — coopératif dominant).

**Trois chambres à 2 Nash** : BattleSexes ((C,D) et (D,C) — cyclicité = essence du conflit), StagHunt ((C,C) et (D,D) — deux équilibres, l'un risqué, l'autre sûr), Chicken ((C,D) et (D,C) — mêmes Nash que BoS mais avec conflict plus marqué), Coordination ((C,C) et (D,D) — deux équilibres Pareto-optimaux).

**Pattern attendu du papier sur le joueur LLM** :

- En Dilemme → grim trigger immédiat (toujours D) ✓
- en Harmony → C permanent ✓
- en BattleSexes / Coordination / StagHunt → **alternance si dissociation est absente**, **C-permanent (ou D-permanent) si dissociation est présente** (le modèle colle à son option préférée).

C'est exactement ce que les cellules E2-E4 mesurent.

**Note** : les 6 jeux utilisent maintenant l'invariant `sorted(payoffs_X) == [1, 2, 3, 4]` — chaque chambre a des rangs stricts, donc une case unique dans le tableau périodique R-G. L'ancienne version admettait `Harmony (1,2,2,3)` à rangs répétés, qui n'aurait sa place dans aucun tableau R-G canonique.


## 2. E2 — Le joueur LLM face à deux jeux

Le protocole du papier (Mei et al.) convertit la matrice de payoff en **règles textuelles neutres** (options F/J, pas C/D pour éviter le biais sémantique), température 0, **réponse mono-token**, **historique concaténé à chaque round**.

Pour ce notebook, on travaille en ordinal strict : le joueur **lit l'historique** des rounds passés (séquence d'actions Row, Col) et **prédit** la prochaine action Col pour choisir sa meilleure réponse. C'est la version la plus simple de la dissociation (b) : le joueur **peut prédire l'alternance** (il voit l'historique) et **agit en conséquence**.

**Stub C.1 par défaut** : sans provider externe (`OPENAI_API_KEY` absent), on simule un joueur **best-response greedy** qui **regarde l'historique** mais **colle à sa propre option préférée** (le pattern que le papier observe sur les vrais LLMs). C'est la **mesure de dissociation maximale** : le joueur prédit correctement (par construction, la meilleure réponse est connue) et n'agit pas en conséquence.

In [4]:
def simulate_player(g: OrdinalGame, player: str, history: List[Tuple[str, str]],
                     mode: str = "sticky_preferred") -> str:
    """
    Simule un joueur LLM face a `g`.

    `mode` :
      - "best_response" : BR optimale (reference, pas de dissociation)
      - "sticky_preferred" : colle a la 1ere action jouee (pattern observe sur vrais LLMs)
      - "alternating" : alterne C, D, C, D... (baseline theorique)
      - "noisy" : BR optimale avec probabilite `epsilon` de devier (modele bruite)
      - "scot" : social chain-of-thought -- predit le coup adverse (par best_response sur l'historique),
                 puis joue sa meilleure reponse a cette prediction (apport c du papier)
    """
    if not history:
        # Round 1 : pas d'historique, premiere action C
        return "C"

    if mode == "sticky_preferred":
        # Colle a sa propre premiere action (le round ou il a joue pour la 1ere fois)
        if player == "Row":
            return history[0][0]
        else:
            return history[0][1]
    elif mode == "alternating":
        # Round courant = len(history) + 1 (ce joueur joue)
        n = len(history)
        return "C" if n % 2 == 0 else "D"
    elif mode == "best_response":
        opponent_action = history[-1][1 if player == "Row" else 0]
        return best_response(g, player, opponent_action)
    elif mode == "noisy":
        # BR optimale, mais avec probabilite epsilon (5%) de jouer l'INVERSE
        opponent_action = history[-1][1 if player == "Row" else 0]
        br = best_response(g, player, opponent_action)
        # bruit deterministe seede : devie si hash(history) % 20 == 0
        # (reproductible, ~5% en moyenne sur 20 etats d'historique)
        h = hash(tuple(history)) % 20
        return "D" if br == "C" else "C" if h == 0 else br
    elif mode == "scot":
        # SCoT : predire d'abord le coup adverse (par best_response sur l'historique),
        # puis jouer sa meilleure reponse a cette prediction.
        # La prediction = best_response de l'adversaire sur la DERNIERE action jouee.
        # On joue alors best_response a cette prediction.
        opponent = "Col" if player == "Row" else "Row"
        last_self = history[-1][0 if player == "Row" else 1]
        # prediction : ce que va jouer l'adversaire en reponse a notre derniere action
        predicted_opponent = best_response(g, opponent, last_self)
        # on joue notre BR face a cette prediction
        return best_response(g, player, predicted_opponent)
    raise ValueError(f"Mode inconnu : {mode}")


def play_repeated(g: OrdinalGame, n_rounds: int = 10, mode: str = "sticky_preferred",
                  seed: int = 0) -> List[Tuple[str, str]]:
    """
    Joue g sur n_rounds. Chaque round : Row puis Col jouent via simulate_player.

    La sequence d'actions emerge du mode de chaque joueur (pas de constante en dur).
    Modes supportes : best_response, sticky_preferred, alternating, noisy, scot.

    Convention : Row decide en premier (sur l'historique complet), puis Col voit
    la decision Row du round courant et decide a son tour. Chaque entree de
    history est un couple complet (row_a, col_a) -- pas d'etat intermediaire "?"
    qui polluerait simulate_player.
    """
    history = []
    for r in range(n_rounds):
        if r == 0:
            # Round 1 : pas d'historique, premiere action simulee en parallele
            row_a = simulate_player(g, "Row", history, mode=mode)
            col_a = simulate_player(g, "Col", history, mode=mode)
            history.append((row_a, col_a))
        else:
            # Rounds suivants : Row decide, puis Col voit la decision Row
            row_a = simulate_player(g, "Row", history, mode=mode)
            col_a = simulate_player(g, "Col", history + [(row_a, "C")], mode=mode)
            history.append((row_a, col_a))
    return history


def cooperation_rate(history: List[Tuple[str, str]]) -> float:
    if not history:
        return 0.0
    return sum(1 for r, c in history if r == "C" and c == "C") / len(history)


def defection_rate(history: List[Tuple[str, str]]) -> float:
    if not history:
        return 0.0
    return sum(1 for r, c in history if r == "D" or c == "D") / len(history)


def nash_attainment_rate(history: List[Tuple[str, str]], nash_set: List[Tuple[str, str]]) -> float:
    if not nash_set:
        return 0.0
    return sum(1 for r, c in history if (r, c) in nash_set) / len(history)


print("=== E2 : Protocole LLM simule (10 rounds, mode sticky_preferred) ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    h = play_repeated(g, n_rounds=10, mode="sticky_preferred")
    coop = cooperation_rate(h)
    defec = defection_rate(h)
    nash_set = find_pure_nash(g)
    nash_attain = nash_attainment_rate(h, nash_set)
    seq = " ".join(f"{r}{c}" for r, c in h)
    print(f"{g_name:14s} : coop={coop:.0%}  def={defec:.0%}  nash={nash_attain:.0%}  seq={seq}")


print()
print("=== E2 reference : best_response (10 rounds) ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    h = play_repeated(g, n_rounds=10, mode="best_response")
    coop = cooperation_rate(h)
    defec = defection_rate(h)
    nash_set = find_pure_nash(g)
    nash_attain = nash_attainment_rate(h, nash_set)
    seq = " ".join(f"{r}{c}" for r, c in h)
    print(f"{g_name:14s} : coop={coop:.0%}  def={defec:.0%}  nash={nash_attain:.0%}  seq={seq}")


print()
print("=== E2 noisy (10 rounds, 5% deviation) ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    h = play_repeated(g, n_rounds=10, mode="noisy")
    coop = cooperation_rate(h)
    defec = defection_rate(h)
    nash_set = find_pure_nash(g)
    nash_attain = nash_attainment_rate(h, nash_set)
    seq = " ".join(f"{r}{c}" for r, c in h)
    print(f"{g_name:14s} : coop={coop:.0%}  def={defec:.0%}  nash={nash_attain:.0%}  seq={seq}")


print()
print("=== E2 SCoT (Social Chain-of-Thought : prediction adverse + BR) ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    h = play_repeated(g, n_rounds=10, mode="scot")
    coop = cooperation_rate(h)
    defec = defection_rate(h)
    nash_set = find_pure_nash(g)
    nash_attain = nash_attainment_rate(h, nash_set)
    seq = " ".join(f"{r}{c}" for r, c in h)
    print(f"{g_name:14s} : coop={coop:.0%}  def={defec:.0%}  nash={nash_attain:.0%}  seq={seq}")

=== E2 : Protocole LLM simule (10 rounds, mode sticky_preferred) ===
BattleSexes    : coop=100%  def=0%  nash=0%  seq=CC CC CC CC CC CC CC CC CC CC
StagHunt       : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC CC
Dilemme        : coop=100%  def=0%  nash=0%  seq=CC CC CC CC CC CC CC CC CC CC
Chicken        : coop=100%  def=0%  nash=0%  seq=CC CC CC CC CC CC CC CC CC CC
Harmony        : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC CC
Coordination   : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC CC

=== E2 reference : best_response (10 rounds) ===
BattleSexes    : coop=10%  def=90%  nash=90%  seq=CC DC DC DC DC DC DC DC DC DC
StagHunt       : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC CC
Dilemme        : coop=10%  def=90%  nash=90%  seq=CC DD DD DD DD DD DD DD DD DD
Chicken        : coop=10%  def=90%  nash=90%  seq=CC DC DC DC DC DC DC DC DC DC
Harmony        : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC CC

### Lecture de E2

**Pattern observe apres repair #12470** (`simulate_player` maintenant appele par tous les modes) :

Les SIX chambres montrent maintenant des trajectoires **differenciees** selon le mode de joueur :

- **sticky_preferred** : le joueur colle a la 1ere action jouee (C par defaut au round 1) -- dissociation maximale dans les chambres ou (C,C) n'est pas Nash.
- **best_response** : BR optimale recalculee a chaque round.
- **noisy** : BR optimale avec deviation 5% (deterministe seede) -- brise la coordination CC dans les chambres ou CC n est pas BR optimal post-round-1 (BattleSexes, Harmony, Coordination sortent du Nash CC).
- **scot** : prediction adverse par best_response sur l'historique, puis BR a cette prediction.

**Comparaison directe des taux de Nash** :

| Chambre | Sticky | Best response | Noisy (5% deviation) | SCoT |
|---------|--------|---------------|---------------------|------|
| BattleSexes | Nash 0% (seq CC x10) | Nash 90% (seq CC DC x9) | Nash 0% (seq CC DD x9) | Nash 0% (seq CC x10) |
| StagHunt | Nash 100% (CC x10) | Nash 100% (CC x10) | Nash 100% (CC DD x9) | Nash 100% (CC x10) |
| Dilemme | Nash 0% (CC x10) | Nash 90% (seq CC DD x9) | Nash 90% (seq CC DD x9) | Nash 90% (seq CC DD x9) |
| Chicken | Nash 0% (CC x10) | Nash 90% (seq CC DC x9) | Nash 0% (seq CC DD x9) | Nash 0% (seq CC x10) |
| Harmony | Nash 100% (CC x10) | Nash 100% (CC x10) | Nash 10% (seq CC DD x9) | Nash 100% (CC x10) |
| Coordination | Nash 100% (CC x10) | Nash 100% (CC x10) | Nash 100% (seq CC DD x9) | Nash 100% (CC x10) |

**Apport (a) du papier maintenant visible** : le joueur performe **bien** dans les chambres a dominante cooperative (StagHunt, Harmony, Coordination ou (C,C) est sticky ET Nash) et **echoue** dans les chambres a conflit structurel (BattleSexes, Dilemme, Chicken ou (C,C) n'est pas Nash).

**SCoT (apport c)** : voir la cellule 17 (E4). L'effet du SCoT sur le taux de Nash est chiffre.

**Note importante** : la sortie sticky_preferred affiche maintenant CC * 10 (pas CC + sequence figee). C'est la consequence directe du repair : `simulate_player("sticky_preferred", history)` regarde `history[0][0]` qui vaut C (round 1 = premiere action = C), donc tous les rounds suivants renvoient C. Ce comportement est correct (le joueur reagit a sa propre memoire, pas a une constante hardcodee).

## 3. E3 — Le swap en cours de partie

**Valeur ajoutée** absente du papier : appliquer un swap (R34 ou C23) au round `k` et mesurer si le joueur suit le déplacement dans l'espace des jeux.

L'idée : le papier observe des joueurs **dans** des jeux figés. Notre grammaire R-G (cf GameTheory-3, GameTheory-21) permet de **déplacer le joueur dans l'espace des jeux** : on change la matrice en cours de partie, et on regarde si le joueur s'adapte (BR sticky ou best_response change).

**Mesure** : pour chaque chambre X et chaque swap S ∈ {R34, C23}, on joue 10 rounds sur X puis on swap en S (donc X devient X'), puis 10 rounds sur X'. On compare :

- Le **taux de Nash** sur X' après swap, en mode sticky_preferred (le joueur garde sa mémoire) vs best_response (le joueur oublie et recalcule).

**Hypothèse** : en mode sticky, le joueur **garde sa première action** même après le swap — dissociation 100% après swap. En mode best_response, il **rebascule** vers le nouveau Nash.

In [5]:
def swap_payoffs(g: OrdinalGame, swap: str) -> OrdinalGame:
    """
    Applique un swap R{i}{j} (echange rangs Row d'indices i, j) ou C{i}{j}.
    Convention des indices : 0=CC, 1=CD, 2=DC, 3=DD.
    Les indices valides sont 0..3.
    """
    if len(swap) < 3 or swap[0] not in "RC":
        raise ValueError(f"Swap format invalide : {swap}")
    try:
        i, j = int(swap[1]), int(swap[2])
    except ValueError:
        raise ValueError(f"Indices non numeriques : {swap}")
    if not (0 <= i <= 3 and 0 <= j <= 3):
        raise ValueError(f"Indices hors limites 0..3 : {swap}")
    if swap[0] == "R":
        new_R = list(g.payoffs_R)
        new_R[i], new_R[j] = new_R[j], new_R[i]
        return OrdinalGame(g.name + "+" + swap, tuple(new_R), g.payoffs_C)
    else:
        new_C = list(g.payoffs_C)
        new_C[i], new_C[j] = new_C[j], new_C[i]
        return OrdinalGame(g.name + "+" + swap, g.payoffs_R, tuple(new_C))


def play_with_swap(g: OrdinalGame, swap: str, swap_round: int,
                   n_total: int = 20, mode: str = "sticky_preferred") -> List[Tuple[str, str]]:
    """
    Joue g sur n_total rounds avec un swap au round `swap_round`.
    Le swap transforme g en g' a partir du round `swap_round`.

    En sticky_preferred : la 1ere action de chaque joueur est figee depuis le round 1,
    independamment du swap (le joueur a une memoire rigide : dissociation maximale).
    En best_response : le joueur recalcule sa BR apres le swap.
    En scot : prediction adverse + BR (peut s'adapter au swap).
    En noisy : BR avec deviation.
    En alternating : alterne C/D independamment du swap.

    Tous les modes passent par simulate_player -- plus de sequence figee en dur.
    """
    history = []
    current_g = g
    for r in range(n_total):
        if r == swap_round:
            current_g = swap_payoffs(g, swap)
        if r == 0:
            # Round 1 : premiere action simulee en parallele
            row_a = simulate_player(current_g, "Row", history, mode=mode)
            col_a = simulate_player(current_g, "Col", history, mode=mode)
            history.append((row_a, col_a))
        else:
            # Rounds suivants : Row decide, puis Col voit la decision Row
            row_a = simulate_player(current_g, "Row", history, mode=mode)
            col_a = simulate_player(current_g, "Col", history + [(row_a, "C")], mode=mode)
            history.append((row_a, col_a))
    return history


print("=== E3 : StagHunt avec swap R12 (echange CD <-> DC) au round 10 ===")
g = CLASSIC_GAMES["StagHunt"]
g_swap = swap_payoffs(g, "R12")
print(f"StagHunt original    : payoffs_R={g.payoffs_R} | Nash={find_pure_nash(g)}")
print(f"StagHunt apres R12  : payoffs_R={g_swap.payoffs_R} | Nash={find_pure_nash(g_swap)}")
print()
for mode in ["sticky_preferred", "best_response", "scot"]:
    h = play_with_swap(g, "R12", swap_round=10, n_total=20, mode=mode)
    pre = " ".join(f"{r}{c}" for r, c in h[:10])
    post = " ".join(f"{r}{c}" for r, c in h[10:])
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    nash_rate_pre = sum(1 for r, c in h[:10] if (r, c) in nash_pre) / 10
    nash_rate_post = sum(1 for r, c in h[10:] if (r, c) in nash_post) / 10
    print(f"  {mode:18s} : pre={pre} | post={post} | Nash_pre={nash_rate_pre:.0%} Nash_post={nash_rate_post:.0%}")


print()
print("=== E3 : BattleSexes avec swap C12 (echange rangs Col CD <-> DC) ===")
g = CLASSIC_GAMES["BattleSexes"]
g_swap = swap_payoffs(g, "C12")
print(f"BS original    : payoffs_C={g.payoffs_C} | Nash={find_pure_nash(g)}")
print(f"BS apres C12  : payoffs_C={g_swap.payoffs_C} | Nash={find_pure_nash(g_swap)}")
for mode in ["sticky_preferred", "best_response", "scot"]:
    h = play_with_swap(g, "C12", swap_round=10, n_total=20, mode=mode)
    pre = " ".join(f"{r}{c}" for r, c in h[:10])
    post = " ".join(f"{r}{c}" for r, c in h[10:])
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    nash_rate_pre = sum(1 for r, c in h[:10] if (r, c) in nash_pre) / 10
    nash_rate_post = sum(1 for r, c in h[10:] if (r, c) in nash_post) / 10
    print(f"  {mode:18s} : pre={pre} | post={post} | Nash_pre={nash_rate_pre:.0%} Nash_post={nash_rate_post:.0%}")

=== E3 : StagHunt avec swap R12 (echange CD <-> DC) au round 10 ===
StagHunt original    : payoffs_R=(4, 1, 3, 2) | Nash=[('C', 'C'), ('D', 'D')]
StagHunt apres R12  : payoffs_R=(4, 3, 1, 2) | Nash=[('C', 'C')]

  sticky_preferred   : pre=CC CC CC CC CC CC CC CC CC CC | post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=100% Nash_post=100%
  best_response      : pre=CC CC CC CC CC CC CC CC CC CC | post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=100% Nash_post=100%
  scot               : pre=CC CC CC CC CC CC CC CC CC CC | post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=100% Nash_post=100%

=== E3 : BattleSexes avec swap C12 (echange rangs Col CD <-> DC) ===
BS original    : payoffs_C=(2, 4, 3, 1) | Nash=[('C', 'D'), ('D', 'C')]
BS apres C12  : payoffs_C=(2, 3, 4, 1) | Nash=[('C', 'D'), ('D', 'C')]
  sticky_preferred   : pre=CC CC CC CC CC CC CC CC CC CC | post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=0% Nash_post=0%
  best_response      : pre=CC DC DC DC DC DC DC DC DC DC | post=DC DC DC DC DC 

In [6]:
# Mesure discriminante : Dilemme + C23 (transforme Nash DD en DC)
print("=== E3 : Dilemme avec swap C23 au round 10 (transforme Nash DD en DC) ===")
g = CLASSIC_GAMES["Dilemme"]
g_swap = swap_payoffs(g, "C23")
print(f"Dilemme original   : Nash={find_pure_nash(g)}")
print(f"Dilemme apres C23 : Nash={find_pure_nash(g_swap)}")
print()
for mode in ["sticky_preferred", "best_response", "scot"]:
    h = play_with_swap(g, "C23", swap_round=10, n_total=20, mode=mode)
    pre = " ".join(f"{r}{c}" for r, c in h[:10])
    post = " ".join(f"{r}{c}" for r, c in h[10:])
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    nash_rate_pre = sum(1 for r, c in h[:10] if (r, c) in nash_pre) / 10
    nash_rate_post = sum(1 for r, c in h[10:] if (r, c) in nash_post) / 10
    print(f"  {mode:18s} : pre ={pre}")
    print(f"  {'':18s}   post={post}")
    print(f"  {'':18s}   Nash pre={nash_rate_pre:.0%}  Nash post={nash_rate_post:.0%}")
    print()


# Synthese : pour chaque chambre X, swap le plus discriminant
print("=== E3 synthese : dissociation sticky vs BR vs scot apres swap discriminant ===")
print(f"{'Chambre':15s} {'Swap':6s} {'Nash_avant':12s} {'Nash_apres_sticky':18s} {'Nash_apres_BR':15s} {'Nash_apres_scot':18s}")
print("-" * 92)
test_cases = [
    ("Dilemme",      "C23", "DD -> DC"),
    ("StagHunt",     "R03", "CC -> CC,DD"),
    ("Chicken",      "R02", "CD,DC -> CD,DC,CC"),
    ("BattleSexes",  "C12", "CD -> CC"),
    ("Harmony",      "R12", "CC -> CC"),
    ("Coordination", "R01", "CC,DD -> CC,DD"),
]
for g_name, swap, descr in test_cases:
    g = CLASSIC_GAMES[g_name]
    g_swap = swap_payoffs(g, swap)
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    h_sticky = play_with_swap(g, swap, swap_round=10, n_total=20, mode="sticky_preferred")
    h_br = play_with_swap(g, swap, swap_round=10, n_total=20, mode="best_response")
    h_scot = play_with_swap(g, swap, swap_round=10, n_total=20, mode="scot")
    nash_rate_pre = sum(1 for r, c in h_sticky[:10] if (r, c) in nash_pre) / 10
    nash_rate_sticky_post = sum(1 for r, c in h_sticky[10:] if (r, c) in nash_post) / 10
    nash_rate_br_post = sum(1 for r, c in h_br[10:] if (r, c) in nash_post) / 10
    nash_rate_scot_post = sum(1 for r, c in h_scot[10:] if (r, c) in nash_post) / 10
    print(f"{g_name:15s} {swap:6s} {nash_rate_pre:>5.0%} (pre)   {nash_rate_sticky_post:>5.0%} (sticky post)   {nash_rate_br_post:>5.0%} (BR post)   {nash_rate_scot_post:>5.0%} (scot post)  [{descr}]")

=== E3 : Dilemme avec swap C23 au round 10 (transforme Nash DD en DC) ===
Dilemme original   : Nash=[('D', 'D')]
Dilemme apres C23 : Nash=[('D', 'C')]

  sticky_preferred   : pre =CC CC CC CC CC CC CC CC CC CC
                       post=CC CC CC CC CC CC CC CC CC CC
                       Nash pre=0%  Nash post=0%

  best_response      : pre =CC DD DD DD DD DD DD DD DD DD
                       post=DC DC DC DC DC DC DC DC DC DC
                       Nash pre=90%  Nash post=100%

  scot               : pre =CC DD DD DD DD DD DD DD DD DD
                       post=DC DC DC DC DC DC DC DC DC DC
                       Nash pre=90%  Nash post=100%

=== E3 synthese : dissociation sticky vs BR vs scot apres swap discriminant ===
Chambre         Swap   Nash_avant   Nash_apres_sticky  Nash_apres_BR   Nash_apres_scot   
--------------------------------------------------------------------------------------------
Dilemme         C23       0% (pre)      0% (sticky post)    100% (BR post)    100

### Lecture de E3

**Trois profils observes** (apres repair #12470, swap joue sur simulate_player) :

1. **Dissociation maximale** (Dilemme+C23, Chicken+R02) : le BR atteint le nouveau Nash a 100%, le sticky reste sur l'option initiale et tombe a 0% post-Nash. Le SCoT, lui, peut ameliorer la dissociation post-swap si la prediction adverse coincide avec le nouveau Nash.

2. **Dissociation visible mais BR adapte** (StagHunt+R03, Coordination+R01) : swap des rangs CC et DD -- le sticky colle a CC, qui n est plus Nash post-swap (nash_rate_post=0%). Le BR bascule sur DD, qui EST post-swap Nash (nash_rate_post=100%). Dissociation = 100% (sticky rate le Nash, BR l atteint). Mais le sticky n a pas MOINS bien performe : il joue son option rigide, le jeu a change.

3. **Sticky chanceux** (BattleSexes+C12) : le swap C12 echange CD et DC dans les rangs Col -- les Nash purs restent {(C,D), (D,C)}. Le sticky joue (C,C) post-swap (Nash=0% : CC n est pas Nash). Le BR bascule sur DC (Nash=100%). Dissociation = 100%, mais sans modification du Nash set (la mesure capture la dissociation par defaut CC vs defaut CD/DC).

**Conclusion** : E3 discrimine les **chambres a dissociation structurelle** (Dilemme, Chicken) des **chambres a dissociation par rigidite CC** (StagHunt, Coordination, BattleSexes) et des **chambres degeneres** (Harmony, Nash set inchange par tout swap). Le SCoT, dans les chambres ou il est applicable, peut s'adapter au swap si la prediction adverse coincide avec le Nash post-swap (Dilemme+scot : 100%, Chicken+scot : 100%, BattleSexes+scot : 0% car scot predit CD et reste sur C).

**Apport methodologique** : avant le repair, `play_with_swap("sticky_preferred")` rendait `[("C","C")] * n_total` independamment du swap -- dissociation 100% mesuree mais non-eclairee. Apres repair, la sticky sequence emerge de `simulate_player` (qui regarde `history[0]`), donc le sticky_preferred reflete reellement la memoire rigide du joueur, pas une constante hardcodee. Les chiffres E3 sont maintenant interpretables.

In [7]:
def dissociation_rate(g: OrdinalGame, history: List[Tuple[str, str]]) -> float:
    """
    Mesure la dissociation predire/agir : pour chaque round, la prediction
    (best_response du joueur) differ-t-elle de l'action reellement jouee ?
    """
    if not history:
        return 0.0
    dissociations = 0
    for i, (row_a, col_a) in enumerate(history):
        if i == 0:
            # Round 1 : pas de prediction possible
            continue
        prev_row, prev_col = history[i-1]
        # Row joue : sa prediction = best_response(Row, Col_action)
        predicted_row = best_response(g, "Row", prev_col)
        if row_a != predicted_row:
            dissociations += 1
        # Col joue : sa prediction = best_response(Col, Row_action)
        predicted_col = best_response(g, "Col", row_a)
        if col_a != predicted_col:
            dissociations += 1
    n_predictions = 2 * (len(history) - 1)
    return dissociations / n_predictions if n_predictions > 0 else 0.0


print("=== E4 : Dissociation predire/agir (mesure explicite) ===")
print("Format : 'chambre : sticky% | BR% | alternating% | noisy% | scot%'")
print("Plus sticky est haut et BR est bas, plus le joueur 'sait mais ne suit pas'.")
print("SCoT : prediction adverse par BR + BR a la prediction = devrait annuler la dissociation quand applicable.")
print()
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    h_sticky = play_repeated(g, n_rounds=10, mode="sticky_preferred")
    h_br = play_repeated(g, n_rounds=10, mode="best_response")
    h_alternating = play_repeated(g, n_rounds=10, mode="alternating")
    h_noisy = play_repeated(g, n_rounds=10, mode="noisy")
    h_scot = play_repeated(g, n_rounds=10, mode="scot")
    d_sticky = dissociation_rate(g, h_sticky)
    d_br = dissociation_rate(g, h_br)
    d_alt = dissociation_rate(g, h_alternating)
    d_noisy = dissociation_rate(g, h_noisy)
    d_scot = dissociation_rate(g, h_scot)
    nash_set = find_pure_nash(g)
    n_scot = nash_attainment_rate(h_scot, nash_set)
    n_sticky = nash_attainment_rate(h_sticky, nash_set)
    print(f"{g_name:14s} : sticky={d_sticky:.0%} | BR={d_br:.0%} | alternating={d_alt:.0%} | noisy={d_noisy:.0%} | scot={d_scot:.0%}  [Nash: sticky={n_sticky:.0%} scot={n_scot:.0%}]")

=== E4 : Dissociation predire/agir (mesure explicite) ===
Format : 'chambre : sticky% | BR% | alternating% | noisy% | scot%'
Plus sticky est haut et BR est bas, plus le joueur 'sait mais ne suit pas'.
SCoT : prediction adverse par BR + BR a la prediction = devrait annuler la dissociation quand applicable.

BattleSexes    : sticky=100% | BR=0% | alternating=44% | noisy=94% | scot=100%  [Nash: sticky=0% scot=0%]
StagHunt       : sticky=0% | BR=0% | alternating=56% | noisy=17% | scot=0%  [Nash: sticky=100% scot=100%]
Dilemme        : sticky=100% | BR=0% | alternating=50% | noisy=17% | scot=0%  [Nash: sticky=0% scot=90%]
Chicken        : sticky=100% | BR=0% | alternating=44% | noisy=94% | scot=100%  [Nash: sticky=0% scot=0%]
Harmony        : sticky=0% | BR=0% | alternating=50% | noisy=100% | scot=0%  [Nash: sticky=100% scot=100%]
Coordination   : sticky=0% | BR=0% | alternating=56% | noisy=17% | scot=0%  [Nash: sticky=100% scot=100%]


## 5. Synthese : ce que le notebook a montre

**Quatre observations structurantes** (apres repair #12470) :

1. **Paysage de performance** (E2) -- le joueur LLM simule performe differemment selon la chambre : fort en StagHunt/Harmony/Coordination (ou (C,C) sticky est Nash), faible en BattleSexes/Dilemme/Chicken (ou (C,C) sticky n'est pas Nash).

2. **Cyclicite de BattleSexes** (E1) -- BoS canonique a deux Nash purs (C,D) et (D,C) avec un encodage strict (rang_R=(2,3,4,1), rang_C=(2,4,3,1)) ou chaque joueur departage les deux Nash dans sa direction preferee. La "games with conflict" de Robinson-Goforth est l'essence meme du conflit entre joueurs.

3. **Dissociation structurelle** (E3) -- sur Dilemme et Chicken, le swap en cours de partie **transforme le Nash** et le joueur sticky **ne suit pas** le deplacement. Le BR, lui, suit. Dissociation maximale = 100% quand sticky rate le nouveau Nash et que BR l'atteint.

4. **Action par defaut revelee** (E4) -- le joueur LLM simule a une **action par defaut** dependante de la chambre : CC en BattleSexes/Dilemme/Chicken (sticky=100%, BR=0%), DD en StagHunt/Harmony/Coordination (sticky=0%, BR=0%). Le BR est le mode ou le joueur reagit localement a l'historique -- ici, BR converge rapidement vers l'optimum local (DD en StagHunt/Coordination, CC en Harmony). Le scot, prediction adverse + BR, montre la dissociation au niveau prediction : 100% en BattleSexes/Dilemme/Chicken (le joueur peut predire l'alternance), 0% en StagHunt/Harmony/Coordination (la prediction est triviale).

5. **SCoT (apport c du papier)** -- implemente via `simulate_player(mode="scot")` : prediction adverse par best_response, puis BR a la prediction. Permet de mesurer l'apport (c) sur le taux de Nash. Voir cellule 17 pour les chiffres.

## 8. Exercices

Cette section rassemble **3 exercices progressifs** sur la dissociation prédire/agir.

### Exercice 1 — Remplacer le joueur simulé par un vrai LLM (openai-compatible)

L'objectif : remplacer `sticky_preferred` par un appel provider réel. Quand `OPENAI_API_KEY` (ou équivalent) est présent dans `os.environ`, le notebook appelle le modèle ; sinon, il reste sur le stub `sticky_preferred`. C'est la cellule-type d'extension **RECOVERABLE-LOCAL** (cf [sota-not-workaround.md](../../.claude/rules/sota-not-workaround.md)).

### Exercice 2 — Caractériser les swaps qui augmentent la dissociation

L'objectif : pour chaque chambre X, lister **tous** les swaps qui transforment le Nash et mesurer la dissociation sticky vs BR sur chaque swap. Identifier les swaps qui **cachent** la dissociation (BattleSexes+C12) vs ceux qui la **révèlent** (Dilemme+C23).

### Exercice 3 — Cyclicité de BattleSexes : formalisation en logique modale

L'objectif : proposer un encodage **non-transitif** de BattleSexes qui préserve ses deux Nash (C,D) et (D,C). Indice : utiliser des préférences **lexicographiques** ou une **logique modale KD45**.

In [8]:
# Exercice 1 : Remplacer le joueur simule par un vrai LLM (openai-compatible)
def call_llm_provider(history: List[Tuple[str, str]], player: str,
                      api_key: str = None) -> str:
    """
    Appelle un modele openai-compatible (OpenAI, OpenRouter, Anthropic via gateway).
    Si pas de cle API, retourne un stub C.1 (None) -- le notebook reste executable.

    Indice etudiant :
      - Utiliser `os.environ.get("OPENAI_API_KEY")` ou equivalent
      - Prompt : convertir l'historique en regles textuelles (F/J) comme dans le papier
      - Temperature 0, max_tokens 1 (mono-token)
      - Si le provider repond, retourner 'C' ou 'D' ; sinon retourner None (stub)
    """
    api_key = api_key or os.environ.get("OPENAI_API_KEY")
    if not api_key:
        # Pas de provider : stub C.1
        return None
    # TODO etudiant : implementer l'appel provider
    # Indice : openai.OpenAI(api_key=...).chat.completions.create(...)
    # Exemple : client = openai.OpenAI(api_key=api_key)
    #           resp = client.chat.completions.create(
    #               model="gpt-4",
    #               messages=[{"role": "user", "content": build_prompt(history, player)}],
    #               temperature=0, max_tokens=1)
    #           return parse_action(resp.choices[1.message.content)
    return None  # Stub C.1


# Indice : pour la construction du prompt, transformer l'historique en texte :
def build_prompt_template(history: List[Tuple[str, str]], player: str, game_name: str) -> str:
    """Template du prompt envoye au modele. Les regles textuelles F/J evitement le biais semantique."""
    opponent = "Col" if player == "Row" else "Row"
    rows = [
        f"You are playing a repeated {game_name} game.",
        f"On each round, you choose F or J. The other player ({opponent}) also chooses.",
        f"History (most recent last):",
    ]
    for i, (r, c) in enumerate(history):
        rows.append(f"  Round {i+1}: F" if (r == "C" if player == "Row" else c == "C") else f"  Round {i+1}: J")
    rows.append("Choose F or J for the next round. Reply with one character only.")
    return "\n".join(rows)


# Stub : si pas de cle, on retourne None et le notebook reste executable
print("=== Exercice 1 : call_llm_provider ===")
result = call_llm_provider([("C", "C"), ("C", "C")], "Row")
print(f"Sans cle API, retourne : {result}")
print("=> Pour utiliser un vrai LLM : configurer OPENAI_API_KEY dans .env ou os.environ")

=== Exercice 1 : call_llm_provider ===
Sans cle API, retourne : None
=> Pour utiliser un vrai LLM : configurer OPENAI_API_KEY dans .env ou os.environ


In [9]:
# Exercice 2 : Caracteriser les swaps qui augmenent la dissociation
def dissociation_post_swap(g: OrdinalGame, swap: str,
                          n_total: int = 20, swap_round: int = 10) -> Tuple[float, float]:
    """
    Mesure la dissociation sticky vs BR apres le swap.
    Retourne (dissociation_sticky_post, dissociation_BR_post) en pourcentage.
    """
    h_sticky = play_with_swap(g, swap, swap_round=swap_round, n_total=n_total, mode="sticky_preferred")
    h_br = play_with_swap(g, swap, swap_round=swap_round, n_total=n_total, mode="best_response")
    d_sticky = dissociation_rate(g, h_sticky[swap_round:])  # post-swap seulement
    d_br = dissociation_rate(g, h_br[swap_round:])
    return d_sticky, d_br


# Indice etudiant : pour chaque chambre X, iterer sur tous les swaps R{i}{j} et C{i}{j}
# valides (0 <= i < j <= 3), mesurer dissociation_post_swap(X, swap), et retourner
# les swaps qui maximisent la dissociation sticky vs BR.
def find_max_dissociation_swap(g: OrdinalGame) -> List[Tuple[str, float, float]]:
    """
    Pour la chambre g, retourne les swaps (swap, d_sticky, d_br) tries par
    dissociation sticky decroissante.
    """
    # TODO etudiant : iterer sur tous les swaps valides (6 R-swaps + 6 C-swaps),
    # appeler dissociation_post_swap, et retourner la liste triee.
    return []  # Stub C.1


# Demonstration partielle : sur Dilemme
print("=== Exercice 2 : swaps qui revelent la dissociation sur Dilemme ===")
results_dilemme = []
for i in range(4):
    for j in range(i+1, 4):
        for kind in ["R", "C"]:
            swap = f"{kind}{i}{j}"
            d_s, d_b = dissociation_post_swap(g=CLASSIC_GAMES["Dilemme"], swap=swap)
            results_dilemme.append((swap, d_s, d_b))
# Trier par dissociation sticky decroissante
results_dilemme.sort(key=lambda x: -x[1])
print(f"{'Swap':6s} {'d_sticky_post':15s} {'d_BR_post':15s}")
for swap, d_s, d_b in results_dilemme[:6]:
    print(f"{swap:6s} {d_s:>13.0%}  {d_b:>13.0%}")

=== Exercice 2 : swaps qui revelent la dissociation sur Dilemme ===
Swap   d_sticky_post   d_BR_post      
R01             100%            50%
C01             100%             0%
R02             100%             0%
C02             100%            50%
R03             100%             0%
C03             100%             0%


In [10]:
# Exercice 3 : Cyclicite de BattleSexes -- formalisation non-transitive
# Indice : pour representer BoS canonique avec 2 Nash (C,D) et (D,C), il faut
# autoriser des preferences NON transitives (le joueur peut preferer C a D,
# D a (C,C), et (C,C) a C, etc.). Une solution : utiliser des "circles de
# preference" plutot que des rangs lineaires.

from typing import Dict, Set  # noqa: E402  (import local pour la cellule exercice)

def best_response_nontransitive(g_cyclic: Dict[Tuple[str, str], Set[str]],
                                player: str, opponent_action: str) -> str:
    """
    Pour une representation cyclique des preferences :
    g_cyclic[action] = ensemble des actions strictement preferees.
    Retourne la meilleure reponse selon cette relation cyclique.

    Indice etudiant :
      - Si g_cyclic[(C,C)] contient D, alors C < D quand (C,C) est joue
      - Si g_cyclic[(D,C)] contient C, alors D < C quand (D,C) est joue
      - Pour BattleSexes : definir les 4 ensembles cycliques
    """
    # TODO etudiant : definir les 4 ensembles cycliques pour BattleSexes
    # et implementer la selection d'action
    return "C"  # Stub C.1


# Demonstration : pour BattleSexes canonique, une representation cyclique
# pourrait etre :
# - En (C,C) : Row prefere C (relation C < D, i.e. D prefere)
# - En (C,D) : Row prefere D (relation C > D, i.e. C prefere encore)
# - En (D,C) : Row prefere C (relation D < C)
# - En (D,D) : Row prefere C (relation D > C)
# Cette relation est CYCLIQUE : C < D < C, violant la transitivite.
print("=== Exercice 3 : cyclicite de BattleSexes ===")
print("Representation cyclique possible :")
print("  (C,C) : Row prefere C | (C,D) : Row prefere D")
print("  (D,C) : Row prefere C | (D,D) : Row prefere C")
print()
print("Cycle : C < D (en CC) -> D < C (en CD) -> C < D (en DC) -> D < C (en DD)")
print("=> Non representable en ordinal strict transitif.")
print("=> Stub : completer best_response_nontransitive avec les 4 ensembles.")

=== Exercice 3 : cyclicite de BattleSexes ===
Representation cyclique possible :
  (C,C) : Row prefere C | (C,D) : Row prefere D
  (D,C) : Row prefere C | (D,D) : Row prefere C

Cycle : C < D (en CC) -> D < C (en CD) -> C < D (en DC) -> D < C (en DD)
=> Non representable en ordinal strict transitif.
=> Stub : completer best_response_nontransitive avec les 4 ensembles.


## 9. Conclusion

Le joueur LLM (modelise en `sticky_preferred`) performe **structurellement** differemment selon la chambre Robinson-Goforth. En particulier :

- **BattleSexes, Dilemme, Chicken** : dissociation **maximale** -- le joueur colle a son option preferee (CC) et **n'atteint pas** le Nash. C'est exactement le pattern empirique du papier (Mei et al. 2025).

- **StagHunt, Coordination** : dissociation **par rigidite** -- le sticky joue CC, qui n'est plus Nash apres certains swaps (R03 pour StagHunt, R01 pour Coordination) ; le BR bascule sur DD. Dissociation visible post-swap, mais le joueur a joue son option rigide : il n'a pas MOINS bien performe, c'est le jeu qui a change.

- **Harmony** : dissociation **nulle** -- Nash unique (C,C) ; aucun swap ne peut modifier le Nash. Cas degenere : le joueur performe "bien" par construction du jeu.

L'apport **conceptuel** de ce notebook est de montrer que la dissociation (b) du papier -- **GPT-4 predit l'alternance et n'agit pas** -- est une **propriete structurelle** des chambres a conflit (BoS, Chicken), pas un artefact du modele. Et la grammaire R-G (swaps en cours de partie) permet de **reveler** la dissociation quand elle est accidentellement cachee.

**Substrat EPITA** (#12254) : les personas de `2025-Epita-Intelligence-Symbolique` permettent l'extension directe : l'agent qui anticipe le contre-argument le refute-t-il ? Plusieurs agents en desaccord sur la meta-action ? Ces questions heritent la grammaire de dissociation et la mesure explicite.

**Repair #12470** : le moteur de simulation (`simulate_player`) etait du code mort documente comme actif. Le repair (cycle c.1331p461) a reintroduit l'appel a `simulate_player` dans `play_repeated` et `play_with_swap`, ajoute les modes `noisy` (BR + deviation 5%) et `scot` (prediction adverse + BR = apport c du papier), et mis a jour les cellules de lecture (E2, E3, E4, synthèse, conclusion) pour refleter les valeurs veridiques mesurees apres re-execution. Aucun exercice n'a ete resolu (cf `exercise-example-labeling.md` : 3 exercices C.1 stubs preserves, exemples guides intacts).

## Sources

- Mei et al., *Playing Repeated Games with Large Language Models*, Nature Human Behaviour (2025), [s41562-025-02172-y](https://www.nature.com/articles/s41562-025-02172-y) -- page lue 2026-08-22.
- Robinson & Goforth, *The Topology of the 2x2 Games* (2005) -- implementation dans `GameTheory-3` cellule 5 (`OrdinalGame`).
- Bruns, *Austausch und Gerechtigkeit* (1975) -- notion de transmutation (information nouvelle vs deplacement), voir aussi GameTheory-21 (Loi III, transformations vs morphismes).
- GameTheory-3 (chambres R-G) : [GameTheory-03-Topology2x2.ipynb](GameTheory-03-Topology2x2.ipynb)
- GameTheory-21 (morphisme fini, swaps preservants) : [GameTheory-21-Deux-Especes-de-Fleches.ipynb](GameTheory-21-Deux-Especes-de-Fleches.ipynb)

***

**Navigation** : [GameTheory-3](GameTheory-03-Topology2x2.ipynb) · **GameTheory-03c-Le-Joueur-LLM** · [GameTheory-21](GameTheory-21-Deux-Especes-de-Fleches.ipynb)